### 知识点 1：TF-IDF 文本检索
#### 实现检索器(Retriever)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from rich.console import Console

console = Console()

# 模拟三篇论文摘要
papers = [
    "毫米波雷达利用多普勒特征和卷积神经网络识别手势。",
    "脑电信号通过差分熵和图神经网络识别情绪。",
    "毫米波雷达结合 Transformer 建模手势的时序特征。",
]

question = "毫米波手势识别使用了哪些方法？"

# 文本转换为 TF-IDF 向量
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 3),
)

paper_vectors = vectorizer.fit_transform(papers)
# console.print("paper_vectors:", paper_vectors)
query_vecter = vectorizer.transform([question])
# console.print("query_vecter:", query_vecter)
# TF-IDF 默认L2归一化，点积即余弦相似度
scores = (paper_vectors @ query_vecter.T).toarray().ravel()
# console.print("scores:", scores)

# 获取最相关的前两篇
top_indices = scores.argsort()[::-1][:2]

for idx in top_indices:
    console.print(f"相似度: {scores[idx]:.3f}")
    console.print(f"论文摘要: {papers[idx]}")


相似度: 0.278

论文摘要: 毫米波雷达利用多普勒特征和卷积神经网络识别手势。

相似度: 0.187

论文摘要: 毫米波雷达结合 Transformer 建模手势的时序特征。

### 第二部分：Augmentation + Generation
#### 知识点 2：将检索结果注入 LLM

In [14]:
import os
import asyncio
from rich.console import Console
from rich.markdown import Markdown
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
console = Console()

async def main():
    client = AsyncOpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.getenv(
            "OPENAI_BASE_URL",
            "https://api.deepseek.com",
        ),
    )

     # 模拟已经检索到的论文摘要
    context = """
    [1] 毫米波雷达可利用多普勒特征和卷积神经网络识别手势。
    [2] 毫米波雷达可利用 Transformer 建模手势时序特征。
    """

    question = "毫米波手势识别有哪些技术路线?"

    response = await client.responses.create(
        model=os.getenv("OPENAI_MODEL", "deepseek-flash"),
        instructions=(
            "你是科研助手。仅根据提供的材料回答。"
            "引用材料编号,不要编造论文信息。"
        ),
        input=f"参考材料: \n{context}\n\n 问题: {question}",
    )

    console.print(Markdown(response.output_text))

await main()

根据提供的材料，毫米波手势识别主要有两类技术路线：                                                                 

 1 基于多普勒特征与卷积神经网络（CNN）的路线：利用毫米波雷达的多普勒特征，并通过卷积神经网络进行手势识别[1]。      
 2 基于 Transformer 的时序建模路线：利用 Transformer 对毫米波雷达手势的时序特征进行建模[2]。